# Probe 4 -- serving, penalties, judge, baseline

**Measurement only.** No colony run, no colony code changed. The phaser, `Agent.decide()` and the
judge run through their own code; only the model they are handed is wrapped
(`probes/probe4_serving.py`) to change *how* each call is served:

| knob | values |
|---|---|
| template | `raw` (today) / `chat` (the tokenizer's chat template, thinking off) |
| adapter | `lora` (adapter/) / `base` (plain Qwen3-4B) |
| penalties | `pen` (today's repetition_penalty / no_repeat_ngram) / `nopen` |

Production today is `raw+lora+pen`.

**Before you start:** GPU T4 x2 (or P100), Internet on. For section 5, attach the private
benchmark dataset (the `candidate_only/` export, no keys). Everything lands in
`/kaggle/working/probe/`, and cell 9 zips it as `probe.zip` -- download that.

Rough times on a T4: phaser ~15 min, decide ~45-60 min, judge ~25 min, single ~25 min,
bench ~2 h. Every section appends row by row, so a dead session keeps what it finished;
set `SECTIONS` to re-run only what is missing.

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/arkapravopal04/Swarm_neural_nets.git'
REPO_DIR = '/kaggle/working/Swarm_neural_nets'

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{cmd} failed: {r.stdout} {r.stderr}')
    return r

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    run(['git', 'clone', REPO_URL, REPO_DIR])
run(['git', '-C', REPO_DIR, 'fetch', 'origin', 'main'])
run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'])
os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, 'probes')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('synced to:', run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s']).stdout.strip())
assert os.path.exists('probes/probe4_serving.py'), 'push probes/probe4_serving.py first'

In [ ]:
!pip -q install -U transformers peft accelerate sentence-transformers faiss-cpu sympy

## Config

In [ ]:
OUT_DIR = '/kaggle/working/probe'
SECTIONS = ['phaser', 'decide', 'judge', 'single', 'bench']   # drop any you don't want
PHASER_REPEATS = 2      # per serving condition (8 conditions)
DECIDE_REPEATS = 3      # per agent x condition (3 agents x 8 conditions)
SINGLE_REPEATS = 5
THINK_TOKENS = 4096     # thinking budget for single + bench
BENCH_TIERS = (1, 2, 3, 4)
os.makedirs(OUT_DIR, exist_ok=True)

## Load the model (same base + adapter as the colony notebook)

In [ ]:
import torch, json, glob
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from peft import PeftModel
import peft.tuners.lora.torchao as _lora_torchao
_lora_torchao.is_torchao_available = lambda: False

tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B')
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B', torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base, os.path.join(REPO_DIR, 'adapter'))
model.eval()
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

import probe4_serving as P
print('eos:', tok.eos_token, '| pad:', tok.pad_token)
print('chat template present:', bool(tok.chat_template))

## 1. Phaser -- does it keep the request's figures?
`goal_kept` / `cons_kept`: request figures (of 7) found in the goal / constraints.
`*_unexpl`: numbers that are none of those, nor one step from two of them.

In [ ]:
if 'phaser' in SECTIONS:
    rows = P.run_phaser(model, tok, embedder, OUT_DIR, repeats=PHASER_REPEATS)
    P.print_table(P.summary_phaser(rows))

## 2. decide() -- three agents of the school run
`stopped`: the model ended on its own stop token (otherwise a stop rule or the cap ended it).
`retries`: degenerate draws re-rolled. `kept` / `unexpl` as above.

In [ ]:
if 'decide' in SECTIONS:
    rows = P.run_decide(model, tok, OUT_DIR, repeats=DECIDE_REPEATS)
    P.print_table(P.summary_decide(rows))

## 3. Judge -- same four REPORTs, many times
Two right, two wrong (expect column). Production is sampled at temperature 0.7, so the
10 repeats show how often the same REPORT flips. Greedy rows are one run each.

In [ ]:
if 'judge' in SECTIONS:
    rows = P.run_judge(model, tok, OUT_DIR)
    P.print_table(P.summary_judge(rows))

## 4. Single call -- base Qwen3-4B, chat template, thinking on
The school prompt with no colony. Read the answers in `single.jsonl`.

In [ ]:
if 'single' in SECTIONS:
    rows = P.run_single(model, tok, OUT_DIR, repeats=SINGLE_REPEATS, max_new_tokens=THINK_TOKENS)
    for r in rows:
        print(f"--- rep {r['rep']}: {r['new_tokens']} tokens, capped={r['capped']}, {r['seconds']} s, "
              f"figures kept {r['figures']['kept']}/7, unexplained {r['figures']['unexplained']}")
        print(r['answer'][:1500])

## 5. Benchmark dev items (tiers 1-4), single call, thinking off and on
Needs the attached `candidate_only` export. Writes `answers_nothink.jsonl` and
`answers_think.jsonl`; grade them locally with `hive_grader.py` (keys stay off Kaggle).

In [ ]:
if 'bench' in SECTIONS:
    hits = glob.glob('/kaggle/input/**/main_prompts.jsonl', recursive=True)
    assert hits, 'attach the candidate_only benchmark dataset'
    with open(hits[0], encoding='utf-8-sig') as f:
        prompts = {r['id']: r for r in map(json.loads, filter(str.strip, f))}
    dev = json.load(open('bench_splits.json'))['dev']
    items = [prompts[i] for i in dev if int(i[1]) in BENCH_TIERS]
    print(len(items), 'items')
    P.run_bench(model, tok, items, OUT_DIR, thinking=False, max_new_tokens=1024)
    P.run_bench(model, tok, items, OUT_DIR, thinking=True, max_new_tokens=THINK_TOKENS)
    print('done')

## Download

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/probe', 'zip', OUT_DIR)
print('download /kaggle/working/probe.zip')